# Install libraries

In [3]:
! pip install langchain_neo4j agentmanager langchain_huggingface sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4

**Restart session**

# LLM & Graph DB setup

In [3]:
from google.colab import userdata
from agentmanager import CloudAgentManager
from langchain_neo4j import Neo4jGraph

cloud_agent_manager = CloudAgentManager()

# Prepare LLM
## This we will use when we need to pass the LLM directly
llm = cloud_agent_manager.prepare_llm(
    provider = "groq",
    api_key = userdata.get('GROQ_API_KEY'),
    model_name = "openai/gpt-oss-20b",
    )
## This we will use to invoke LLM
agent, _ = await cloud_agent_manager.prepare_agent(llm)


# Connect to graph DB
graph = Neo4jGraph(
    url = userdata.get('NEO4J_URI'),
    username = userdata.get('NEO4J_USERNAME'),
    password = userdata.get("NEO4J_PASSWORD")
)

# Support functions

In [ ]:
from langchain_neo4j import GraphCypherQAChain
from langchain_neo4j import Neo4jVector
from langchain_huggingface import HuggingFaceEmbeddings

# Prepare query generation & execution pipeline
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)

# Prepare vector retriever
vector_index = Neo4jVector.from_existing_graph(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5"),
    url = userdata.get('NEO4J_URI'),
    username = userdata.get('NEO4J_USERNAME'),
    password = userdata.get("NEO4J_PASSWORD"),
    # search_type="hybrid",
    # keyword_index_name=keyword_index_name,
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)
vector_retriever = vector_index.as_retriever()

In [5]:
# Prepare entity extractor

import json
from typing import List
from pydantic import BaseModel, Field

class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All person, organization, or business entities in the text",
    )

def get_graph_labels() -> str:
    result = graph.query("CALL db.labels() YIELD label RETURN label")

    # Build comma-separated string directly
    return ", ".join(
        row["label"]
        for row in result
        if row["label"] not in {"__Entity__", "__Document__", "Entity", "Document"}
    )


async def entity_extraction(user_query: str, node_labels: str, max_retries: int = 3) -> Entities:
    input_text = f"""From the below text, extract all entities of the following types: {node_labels}.

Return ONLY valid JSON in this format:
{{
  'names': []
}}

Text: {user_query}
"""
    for attempt in range(max_retries):
        print(f"Attempt {attempt + 1}/{max_retries}")

        response = await cloud_agent_manager.get_agent_response(agent, input_text)
        raw_text = response[0].content.strip()

        try:
            # Parse + validate with BaseModel
            parsed = json.loads(raw_text)
            entities = Entities(**parsed)

            # ✅ Stop if entities found
            if entities.names:
                return Entities(**parsed)
        except Exception as e:
            pass
    # If all attempts fail
    print("No entities extracted after max retries.")
    return Entities(names=[])

In [6]:
# Prepare immidiate connections retriever

from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars


def generate_full_text_query(input: str) -> str:
    """
    Prepares a fuzzy 'AND' search string for Neo4j.
    Example: 'Steve Jobs' -> 'Steve~2 AND Jobs~2'
    """
    # Clean the input and split into words
    words = [el for el in remove_lucene_chars(input).split() if el]
    # print(words)

    if not words:
        return ""

    # Join words with AND and add the fuzzy operator ~2
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    # print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


async def immediate_connections_retriever(entities: Entities) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""

    # Iterate through each identified entity
    for entity in entities.names:
        # Query the graph for each entity's relationships
        response = graph.query(
            """
            CALL db.index.fulltext.queryNodes('entity', $query, {limit: 2})
            YIELD node, score

            CALL (node) {
                MATCH (node)-[r]->(neighbor)
                WHERE type(r) <> 'MENTIONS'
                RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output

                UNION ALL

                MATCH (node)<-[r]-(neighbor)
                WHERE type(r) <> 'MENTIONS'
                RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
            }

            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )

        # 4. Join the results into a single string block
        result += "\n" + "\n".join([el['output'] for el in response])

    return result

In [7]:
async def review_answer(user_query: str, answer: str) -> bool:

    fallback_needed = False

    review_prompt = f"""You are an answer relevance evaluator.

If the Answer contains any information relevant to the Question, return exactly: ok
Otherwise return exactly: not ok

Return exactly: ok
if the Answer contains any meaningful, specific information relevant to the Question.

Return exactly: not ok
if the Answer is vague, restates the question, or provides no real information.

Return only:
- ok
- not ok

Question:
{user_query}

Answer:
{answer}
"""
    # with ls.tracing_context(project_name="graphRAG", enabled=True):
    review_response = await cloud_agent_manager.get_agent_response(agent, review_prompt)
    review_result = review_response[0].content.strip().lower()
    print(f"Review Result: {review_result}\n")

    # If review result is ok, then fallback not needed.
    # If review result is not ok, then fallback needed.
    if review_result != "ok":
        fallback_needed = True

    return fallback_needed

# Prerequisite

In [8]:
node_labels = get_graph_labels()
print(node_labels)

Person, Object
